# Sleepy Driver - ML for Simple EEG Data

Everything up to preprocessing is done already so you don't have to do it again.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## Starter Code

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [3]:

data_raw = pd.read_csv("sleepy_driver/data/acquiredDataset.csv")
datas0 = data_raw[data_raw['classification'] == 0]
datas1 = data_raw[data_raw['classification'] == 1]

def create_windows(data, window_size=5):
    windows = []
    labels = []
    for i in range(len(data) - window_size + 1):
        window = data.iloc[i:i + window_size].drop('classification', axis=1)
        label = data.iloc[i + window_size - 1]['classification']
        windows.append(np.array(window))
        labels.append(label)
    return windows, labels

w0, l0 = create_windows(datas0)
w1, l1 = create_windows(datas1)

w = np.array(w0 + w1)
l = l0 + l1

w.shape, len(l) # should output (3727, 5, 10), 3727

((3727, 5, 10), 3727)

In [4]:
# split into 80/10/10 train/val/test split
X_train, X_temp, y_train, y_temp = train_test_split(w, l, test_size=0.2, random_state=42, stratify=l)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# normalize data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_valid = scaler.transform(X_valid.reshape(-1, X_valid.shape[-1])).reshape(X_valid.shape)
X_test = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

print(X_train.shape, len(y_train))
print(X_valid.shape, len(y_valid))
print(X_test.shape, len(y_test))

(2981, 5, 10) 2981
(373, 5, 10) 373
(373, 5, 10) 373


## All you now

In [5]:
# create a pytorch dataset class that takes the windows and labels as parameters 
class EEGDataset(Dataset):
    def __init__(self, windows, labels):
        self.windows = windows
        self.labels = labels
    

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return torch.tensor(self.windows[idx]), torch.tensor(self.labels[idx])


train_dataset = EEGDataset(X_train, y_train)
val_dataset = EEGDataset(X_valid, y_valid)
test_dataset = EEGDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [9]:
# construct a simple model for eval
class MultilayerPerceptron(nn.Module):
    def __init__(self, input_size=50, hidden_size=16, output_size=1):
        super().__init__()
        self.flatten = nn.Flatten()
        self.batchnorm = nn.BatchNorm1d(input_size)
        self.linear_stack= nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
            nn.Dropout(0.5)
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.batchnorm(x)
        x = self.linear_stack(x)
        return x

model = MultilayerPerceptron()
model = model.to(device)
loss = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
epochs = 100
# instantiate model, loss function (BCEWithLogitsLoss), optimizer(Adam), and set epochs

In [10]:
# train and validate function
def train_and_validate(model, train_loader, valid_loader, criterion, optimizer, epochs):
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.float().to(device), y.float().unsqueeze(1).to(device)
            y_pred = model(x)
            loss = criterion(y_pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in valid_loader:
                x, y = x.float().to(device), y.float().unsqueeze(1).to(device)
                y_pred = model(x)
                val_loss += criterion(y_pred, y).item()
        print(f"Epoch {epoch+1}/{epochs}, Validation Loss: {val_loss/len(valid_loader)}")
train_and_validate(model, train_loader, val_loader, loss, optimizer, epochs)

Epoch 1/100, Validation Loss: 0.5574196601907412
Epoch 2/100, Validation Loss: 0.5130067442854246
Epoch 3/100, Validation Loss: 0.5319920480251312
Epoch 4/100, Validation Loss: 0.5363480672240257
Epoch 5/100, Validation Loss: 0.521846113105615
Epoch 6/100, Validation Loss: 0.5009994159142176
Epoch 7/100, Validation Loss: 0.49284640451272327
Epoch 8/100, Validation Loss: 0.4715445364514987
Epoch 9/100, Validation Loss: 0.48549171288808185
Epoch 10/100, Validation Loss: 0.4962750971317291
Epoch 11/100, Validation Loss: 0.47600841025511426
Epoch 12/100, Validation Loss: 0.46405800928672153
Epoch 13/100, Validation Loss: 0.4866582279404004
Epoch 14/100, Validation Loss: 0.44722604254881543
Epoch 15/100, Validation Loss: 0.46188730994860333
Epoch 16/100, Validation Loss: 0.45047332098086673
Epoch 17/100, Validation Loss: 0.504106804728508
Epoch 18/100, Validation Loss: 0.4652904619773229
Epoch 19/100, Validation Loss: 0.4585965648293495
Epoch 20/100, Validation Loss: 0.4292083705464999
Epoc

In [11]:
# build evaluation function
def evaluate_model(model, dataloader, criterion):
  model.eval()
  loss = 0.0
  correct = 0
  total = 0

  predlist = []
  ylist = []

  for i, (X,y) in enumerate(dataloader):
    with torch.inference_mode():
      X = X.float().to(device)
      y = y.float().unsqueeze(1).to(device)
      y_pred = model(X)
      batch_loss = criterion(y_pred, y)
      loss += batch_loss.item()
      predlist.extend(torch.round(torch.sigmoid(y_pred)).cpu().numpy())
      ylist.extend(y.cpu().numpy())
      correct += (torch.round(torch.sigmoid(y_pred)) == y).sum().item()
      total += y.size(0)

  return {
      'model_name' : type(model).__name__,
      'loss' : loss,
      'acc' : round(correct / total,4),
    }, classification_report(ylist, predlist)

test_results, report = evaluate_model(model, test_loader, loss)